In [1]:
import numpy as np
import pandas as pd
df = pd.read_csv("/Users/stephenwillis/Desktop/archive/cleaned_loan_data_dev.csv")

In [2]:
df.head(5)

,Id,City,State,Zip,Bank,BankState,NAICS,ApprovalDate,ApprovalFY,Term,NoEmp,FranchiseCode,UrbanRural,RevLineCr,LowDoc,Defaulted,GrAppv,SBA_Appv,ExistingBusiness
0,5767963003,OHIOPYLE,PA,15470,BNY MELLON NATL ASSOC,PA,313311,1993-07-09,1993,75,45,1,0,0.0,0.0,0.0,67000.0,56950.0,1.0
1,7554323005,ENGLEWOOD,CO,80111,WELLS FARGO BANK NATL ASSOC,CO,0,1994-09-07,1994,84,23,1,0,0.0,1.0,0.0,50000.0,45000.0,1.0
2,1999655008,SELKIRK,NY,12158,FIRST NIAGARA BANK NATL ASSOC,NY,339932,2006-09-01,2006,84,7,0,2,0.0,0.0,0.0,65000.0,32500.0,1.0
3,7897023000,NEW YORK,NY,10012,EMPIRE ST. CERT. DEVEL CORP,NY,0,1994-12-12,1995,240,50,1,0,0.0,0.0,0.0,209000.0,209000.0,1.0
4,3381625007,Central point,OR,97502,PEOPLE'S BANK OF COMMERCE,OR,236118,2009-04-16,2009,12,2,0,2,1.0,NaN,0.0,20000.0,17000.0,1.0


In [3]:
# Set the loan identifier to be the index
df = df.set_index("Id")
# Now before we go any further, we will split our data set into train and test, given that we do not have a separate test set
from sklearn.model_selection import train_test_split
X = df.drop(axis = 1, columns = ["Defaulted"]) 
y = df["Defaulted"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

In [4]:
X_train.columns
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 726705 entries, 6383304006 to 2277345001
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   City              726677 non-null  object 
 1   State             726695 non-null  object 
 2   Zip               726705 non-null  int64  
 3   Bank              725489 non-null  object 
 4   BankState         725482 non-null  object 
 5   NAICS             726705 non-null  int64  
 6   ApprovalDate      726705 non-null  object 
 7   ApprovalFY        726705 non-null  int64  
 8   Term              726705 non-null  int64  
 9   NoEmp             726705 non-null  int64  
 10  FranchiseCode     726705 non-null  int64  
 11  UrbanRural        726705 non-null  int64  
 12  RevLineCr         502273 non-null  float64
 13  LowDoc            721817 non-null  float64
 14  GrAppv            726705 non-null  float64
 15  SBA_Appv          726705 non-null  float64
 16  ExistingBusi

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler

class WOEEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, features, eps=1e-4):
        self.features = features
        self.eps = eps
        self.woe_maps_ = {}

    def fit(self, X, y):
        for col in self.features:
            df = pd.concat([X[col], y], axis=1)
            grouped = df.groupby(col)[y.name].agg(['sum', 'count'])
            grouped["nevent"] = grouped["count"] - grouped["sum"]
            event_dist = grouped["sum"] / grouped["sum"].sum()
            nevent_dist = grouped["nevent"] / grouped["nevent"].sum()
            self.woe_maps_[col] = np.log( (event_dist + self.eps) /
                                         (nevent_dist + self.eps) ).to_dict()
        return self

    def transform(self, X):
        X_enc = X.copy()
        for col, mapping in self.woe_maps_.items():
            X_enc[col] = X_enc[col].map(mapping)
        return X_enc

    def set_output(self, *, transform=None):
        return self


num_cols = ["Term", "NoEmp", "ApprovalFY", "UrbanRural", "RevLineCr", 
            "LowDoc","GrAppv", "SBA_Appv", "ExistingBusiness"]

cat_cols = [col for col in X_train.columns if col not in num_cols]

num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy='median')),
    ("scale", RobustScaler())
])

cat_pipe = Pipeline([
    ("impute", SimpleImputer(strategy='most_frequent')),
    ("encode", WOEEncoder(features=cat_cols)),
    ("fill",    SimpleImputer(strategy="constant", fill_value=0)),  # FIXING UNSEEN WOE ERRORS
])


preproc = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
    ], remainder = "drop")

preproc.set_output(transform="pandas")

# we need to:
### drop nan (or impute)
### turn catagorical numerical
### standard scale maybe
### then apply gridsearch CV or we can build other models with up smapling down sampling etc



ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('impute',
                                                  SimpleImputer(strategy='median')),
                                                 ('scale', RobustScaler())]),
                                 ['Term', 'NoEmp', 'ApprovalFY', 'UrbanRural',
                                  'RevLineCr', 'LowDoc', 'GrAppv', 'SBA_Appv',
                                  'ExistingBusiness']),
                                ('cat',
                                 Pipeline(steps=[('impute',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encode',
                                                  WOEEncoder(features=['City',
                                                                       'State',
                                                                       'Zip',
                                                                       'Bank',
                                                                       'BankState',
                                                                       'NAICS',
                                                                       'ApprovalDate',
                                                                       'FranchiseCode'])),
                                                 ('fill',
                                                  SimpleImputer(fill_value=0,
                                                                strategy='constant'))]),
                                 ['City', 'State', 'Zip', 'Bank', 'BankState',
                                  'NAICS', 'ApprovalDate', 'FranchiseCode'])])

In [6]:
# here we can make sure that our above pipeline works properly with 
# a simple Logistic Regression example
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_recall_curve

# Play with model as needed
logistic_model = LogisticRegression()

totality = Pipeline([
    ("pre", preproc),
    ("model", logistic_model)
]) 

totality.fit(X_train, y_train)


y_proba = totality.predict_proba(X_test)[:,1]
ap = average_precision_score(y_test, y_proba)
print(f"Average Precision: {ap:.4f}")

Average Precision: 0.6476


In [7]:
# time to build a cv pipeline to make some tests

kfold = StratifiedKFold(n_splits=5, shuffle = True, random_state = 42)

logistic_estimator = Pipeline([
    ("pre", preproc),
    ("model", LogisticRegression()) 
]) 

logistic_param_grid = {
    "model__penalty": ["l2"],
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100]
}

gscv = GridSearchCV(
    estimator = logistic_estimator, 
    param_grid = logistic_param_grid, 
    scoring = "average_precision",
    n_jobs = -1,
    refit = True,
    cv = kfold,
    return_train_score = True
)

gscv.fit(X_train, y_train)

print(f"Best params: {gscv.best_params_}")
print(f"Best CV Average Precision: {gscv.best_score_}")

Best params: {'model__C': 0.001, 'model__penalty': 'l2'}
Best CV Average Precision: 0.6399125472467863


In [8]:
# This result doesnt seem great, we need to investigate further
print(X_train[num_cols].describe())

# We see the presence of outliers is hurting us
#import seaborn as sns
#sns.boxplot(data = X_train[["Term","NoEmp"]])

                Term          NoEmp     ApprovalFY     UrbanRural  \
count  726705.000000  726705.000000  726705.000000  726705.000000   
mean      110.848424      11.406846    2001.137451       0.756948   
std        78.884056      72.528465       5.914977       0.646394   
min         0.000000       0.000000    1966.000000       0.000000   
25%        60.000000       2.000000    1997.000000       0.000000   
50%        84.000000       4.000000    2002.000000       1.000000   
75%       120.000000      10.000000    2006.000000       1.000000   
max       569.000000    9999.000000    2014.000000       2.000000   

           RevLineCr         LowDoc        GrAppv      SBA_Appv  \
count  502273.000000  721817.000000  7.267050e+05  7.267050e+05   
mean        0.323762       0.123698  1.929556e+05  1.497076e+05   
std         0.467911       0.329236  2.830346e+05  2.282997e+05   
min         0.000000       0.000000  1.000000e+03  5.000000e+02   
25%         0.000000       0.000000  3.5000

In [9]:
# Let us try handling outliers by use of a special, data-driven transformation
# We also try using a weighted cost function
from sklearn.preprocessing import QuantileTransformer

new_num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy='median')),
    ("quantile", QuantileTransformer(output_distribution='normal', n_quantiles=1000)),
    #alternatively: ("power",    PowerTransformer(method="yeo-johnson"))
    ("scale", RobustScaler()),
])

new_preproc = ColumnTransformer([
    ("num", new_num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
], remainder="drop")

new_preproc.set_output(transform="pandas")

# time to build a cv pipeline to make some tests
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

logistic_estimator = Pipeline([
    ("pre", new_preproc),
    ("model", LogisticRegression(solver="saga", max_iter=1000)) 
]) 

logistic_param_grid = {
    "model__penalty": ["l1", "l2"],
    "model__C": [0.01, 0.1, 1.0, 10.0],
    "model__class_weight": ["balanced", {0:1, 1:2}, {0:1, 1:5}, None]
}

gscv = GridSearchCV(
    estimator=logistic_estimator, 
    param_grid=logistic_param_grid, 
    scoring="average_precision",
    n_jobs=-1,
    refit=True,
    cv=kfold,
    return_train_score=True
)

gscv.fit(X_train, y_train)
print(f"Best params: {gscv.best_params_}")
print(f"Best CV Average Precision: {gscv.best_score_}")

Best params: {'model__C': 0.01, 'model__class_weight': None, 'model__penalty': 'l1'}
Best CV Average Precision: 0.6761085122200963


In [10]:
# we are making marginal gains tuning the Logistic Regression
# rather than continuing in such a fashion we can investigate how RandomForestClassifier fares

In [11]:
# now lets try random forest
# time to build a cv pipeline to make some tests

from sklearn.model_selection import RandomizedSearchCV

kfold = StratifiedKFold(n_splits=5, shuffle = True, random_state = 42)

estimator = Pipeline([
    ("pre", preproc),
    ("model", RandomForestClassifier(n_jobs = -1, max_samples=0.5 )) 
]) 

param_grid = {
    "model__max_features" : ["sqrt", "log2"],
    "model__n_estimators": [50, 100]
}

gscv = GridSearchCV(
    estimator = estimator, 
    param_grid = param_grid, 
    scoring = "average_precision",
    n_jobs = 1,
    refit = True,
    cv = kfold,
    return_train_score = True
)

gscv.fit(X_train, y_train)

print(f"Best params: {gscv.best_params_}")
print(f"Best CV Average Precision: {gscv.best_score_}")

Best params: {'model__max_features': 'log2', 'model__n_estimators': 100}
Best CV Average Precision: 0.901310086910159


In [12]:
# Okay random forest is already outperforming Logistic Regression by quite a bit
# Let us try a different preprocessing approach for the tree based methods

num_cols = ["Term", "NoEmp"]
cat_cols = [col for col in X_train.columns if col not in num_cols]

rf_num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy='constant', fill_value=-1)),
    #("scale", RobustScaler())
])

rf_cat_pipe = Pipeline([
    ("encode_nan", SimpleImputer(strategy='constant', fill_value = "__MISSING__")),
    ("woe_encode", WOEEncoder(features=cat_cols)),
    ("post_fill",    SimpleImputer(strategy="constant", fill_value=0)),  # FIXING UNSEEN WOE ERRORS
])


preproc = ColumnTransformer([
    ("num", rf_num_pipe, num_cols),
    ("cat", rf_cat_pipe, cat_cols)
    ], remainder = "drop")

preproc.set_output(transform="pandas")



kfold = StratifiedKFold(n_splits=5, shuffle = True, random_state = 42)

estimator = Pipeline([
    ("pre", preproc),
    ("model", RandomForestClassifier(n_jobs = -1, max_samples=0.5 )) 
]) 

param_grid = {
    "model__max_features" : ["log2"],
    "model__n_estimators": [100]
}

gscv = GridSearchCV(
    estimator = estimator, 
    param_grid = param_grid, 
    scoring = "average_precision",
    n_jobs = 1,
    refit = True,
    cv = kfold,
    return_train_score = True
)

gscv.fit(X_train, y_train)

print(f"Best params: {gscv.best_params_}")
print(f"Best CV Average Precision: {gscv.best_score_}")


Best params: {'model__max_features': 'log2', 'model__n_estimators': 100}
Best CV Average Precision: 0.8940729789426916


In [13]:
# Testing undersampling
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

#X, y = ...  # Features and binary target
print(f"Before downsampling: {Counter(y_train)}")

rus = RandomUnderSampler(sampling_strategy='auto', random_state=42)
X_res, y_res = rus.fit_resample(X_train, y_train)

print(f"After downsampling: {Counter(y_res)}")

Before downsampling: Counter({0.0: 599197, 1.0: 127508})
After downsampling: Counter({0.0: 127508, 1.0: 127508})


In [14]:
# NOW GOING TO SEE IF UPSAMPLING AND DOWNSAMPLING HELP

In [15]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter
from sklearn.metrics import average_precision_score, make_scorer
from sklearn.model_selection import cross_validate


num_cols = ["Term", "NoEmp"]
cat_cols = [col for col in X_train.columns if col not in num_cols]

rf_num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy='constant', fill_value=-1)),
])

rf_cat_pipe = Pipeline([
    ("encode_nan", SimpleImputer(strategy='constant', fill_value = "__MISSING__")),
    ("woe_encode", WOEEncoder(features=cat_cols)),
    ("post_fill",    SimpleImputer(strategy="constant", fill_value=0)),  # FIXING UNSEEN WOE ERRORS
])

preproc = ColumnTransformer([
    ("num", rf_num_pipe, num_cols),
    ("cat", rf_cat_pipe, cat_cols)
], remainder= "drop")

preproc.set_output(transform="pandas")

undersampled_pipeline = ImbPipeline([
    ("pre", preproc),
    ("under", RandomUnderSampler(sampling_strategy='auto', random_state=42)),
    ("model", RandomForestClassifier(max_samples=0.5, max_features="log2", n_estimators=100))
])

cv_results = cross_validate(
    estimator = undersampled_pipeline,
    X = X_train,
    y = y_train,
    cv = 5,
    scoring = "average_precision",
    return_train_score = True)


print(f"Mean test Average Precision: {cv_results['test_score'].mean():.4f}")
print(f"All test fold scores: {cv_results['test_score']}")

Mean test Average Precision: 0.8813
All test fold scores: [0.88051124 0.88259986 0.88229759 0.88422157 0.87665551]


In [16]:
# Let's now build the pipeline for oversampling
from imblearn.over_sampling import SMOTE, BorderlineSMOTE
from joblib import parallel_backend
from sklearn.model_selection import StratifiedKFold
from collections import defaultdict
import warnings
import os

warnings.filterwarnings("ignore", category=UserWarning)


n_total = os.cpu_count()
n_best = max(1, n_total-1)

best_params = {
    "n_jobs": n_best,
    "max_features":  "log2",
    "max_samples":  0.5,
    "n_estimators":  100
}


rfc = RandomForestClassifier(**best_params)
cfv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)

pipelines = {
    "SMOTE" : ImbPipeline([
        ("preproc", preproc),
        ("SMOTE", SMOTE(random_state = 42)),
        ("model", rfc)
    ]),
    "BSMOTE": ImbPipeline([
        ("preproc", preproc),
        ("BoSMOTE", BorderlineSMOTE(random_state = 42)),
        ("model", rfc)
    ])
}

results = defaultdict(dict)

for name, pipe in pipelines.items(): 
    with parallel_backend("loky"):
        pipe.fit(X_train, y_train)
        y_proba = pipe.predict_proba(X_test)[:,1]
        ap = average_precision_score(y_test, y_proba)
        results[name] = ap #replace with dictionary identifying the given params
        
print(results)
        

    

defaultdict(<class 'dict'>, {'SMOTE': 0.9035350186530098, 'BSMOTE': 0.9028249598452882})


In [17]:
#### Now we can do some hyperparameter tuning for the under/over sampling methods
#SMOTE? #{"sampling_strategy": {"auto", 0.5}, "k_neighbors": {3,7}}
#BorderlineSMOTE? #{sampling_strategy: {"auto", 0.5}, "m_neighbors": {5,10}}
# RandomUnderSampler? {sampling_strategy: {"auto", 0.75, 0.5}}


In [18]:
from sklearn.model_selection import RandomizedSearchCV
RandomizedSearchCV?


pipelines = {
    "SMOTE" : ImbPipeline([
        ("preproc", preproc),
        ("SMOTE", SMOTE(random_state = 42)),
        ("model", rfc)
    ]),
    "BSMOTE": ImbPipeline([
        ("preproc", preproc),
        ("BoSMOTE", BorderlineSMOTE(random_state = 42)),
        ("model", rfc)
    ]),
    "RUS": ImbPipeline([
        ("preproc", preproc),
        ("RUS", RandomUnderSampler(random_state = 42)),
        ("model", rfc)
    ])
}


SMOTE_params = {
    "SMOTE__sampling_strategy": ['auto', 0.5],
    "SMOTE__k_neighbors": [3,8]
}

B_SMOTE_params = {
    "BoSMOTE__sampling_strategy": ['auto', 0.5],
    "BoSMOTE__m_neighbors": [3,8]
}

RUS_params = {
    "RUS__sampling_strategy": ['auto', 0.75, 0.5]
}

search_params = {"SMOTE": SMOTE_params, "BSMOTE": B_SMOTE_params, "RUS": RUS_params}


for name, pipeline in pipelines.items():
    with parallel_backend("loky"):
        rscv = RandomizedSearchCV(
            estimator = pipeline,
            param_distributions = search_params[name],
            scoring = 'average_precision',
            cv = StratifiedKFold(),
            return_train_score = True
        )
        rscv.fit(X_train, y_train)
        print(f"Results for {name}:")
        print(rscv.cv_results_)

Results for SMOTE:
{'mean_fit_time': array([192.83613844, 123.79531198, 195.8552424 , 168.02478762]), 'std_fit_time': array([ 5.50877142, 35.57872012, 20.64830476, 19.34930963]), 'mean_score_time': array([2.48007007, 5.89075084, 4.79584374, 3.08277526]), 'std_score_time': array([0.47241199, 1.25148075, 1.60562756, 1.99199173]), 'param_SMOTE__sampling_strategy': masked_array(data=['auto', 0.5, 'auto', 0.5],
             mask=[False, False, False, False],
       fill_value='?',
            dtype=object), 'param_SMOTE__k_neighbors': masked_array(data=[3, 3, 8, 8],
             mask=[False, False, False, False],
       fill_value=999999), 'params': [{'SMOTE__sampling_strategy': 'auto', 'SMOTE__k_neighbors': 3}, {'SMOTE__sampling_strategy': 0.5, 'SMOTE__k_neighbors': 3}, {'SMOTE__sampling_strategy': 'auto', 'SMOTE__k_neighbors': 8}, {'SMOTE__sampling_strategy': 0.5, 'SMOTE__k_neighbors': 8}], 'split0_test_score': array([0.89255128, 0.89258106, 0.89435304, 0.8939637 ]), 'split1_test_score': 

In [19]:
# No signifigant difference achieved via hyperparameter tuning in this case....